In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# clone the repository

In [ ]:
!git clone --recurse-submodules https://github.com/zash13/CollaborativeTrafficSignal.git
%cd /kaggle/working/CollaborativeTrafficSignal
!chmod +x run.sh

---

# Overwriting the Configuration

#### You can modify the simulation of the program before running the script.  


In [ ]:
%%writefile config.py 
import os

class Config:
    SUMO_BINARY = os.environ.get("SUMO_BINARY", "sumo")
    NET_FILE = os.path.join(os.getcwd(), "my_4way.net.xml")
    ROUTE_FILE = os.path.join(os.getcwd(), "my_4way.rou.xml")
    SUMOCFG_FILE = os.path.join(os.getcwd(), "my_4way.sumocfg")
    SIM_STEP = 1.0
    MAX_STEPS = 600
    MAX_VEHICLES = 200 #i have 12 line , each can have something like 10 car , so 12 * 10 = 120 , 200 is for stress test 
    SPAWN_MIN_INTERVAL = 1.0
    SPAWN_MAX_INTERVAL = 3.0
    DEFAULT_SPEED = 13.89  # m/s (≈50km/h)
    MIN_GREEN_TIME = 10.0
    MAX_WAIT_EXPECTED = 200.0  # Maximum expected waiting time in seconds
    MAX_VEHICLES_PER_LANE = 12


# Overwriting the Training 

#### You can modify the agent or other components of the program before running the script.  
#### Alternatively, you may run the script with its default settings—modifying the repository is not strictly necessary.


In [ ]:
%%writefile train.py 
#!/usr/bin/env python3
from DQN.DQN_Agent import (
    AgentFactory,
    AgentType,
    EpsilonPolicyType,
    EpsilonPolicy,
    UpdateTargetNetworkType,
    RewardPolicyType,
)
import os
import sys
from sumo_env import SumoEnv  
from config import Config
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"[INFO] Using GPU(s): {[gpu.name for gpu in gpus]}")
else:
    print("[WARN] No GPU detected, using CPU.")

EPOCHES = 1


def train_dqn(env, num_episodes=EPOCHES, max_steps_per_episode=Config.MAX_STEPS):
    action_size = len(env.phases[env.tls_ids[0]])
    obs_dim = env.obs_dim
    epsilon_min = 0.1
    epsilon_decay = 0.995
    ep_policy = EpsilonPolicy(
        epsilon_min=epsilon_min,
        epsilon_decay=epsilon_decay,
        policy=EpsilonPolicyType.DECAY,
    )

    agent = AgentFactory.create_agent(
        AgentType.DUELING_DQN,
        action_size=action_size,
        state_size=obs_dim,
        learning_rate=0.0001,
        gamma=0.99,
        epsilon=1.0,
        batch_size=32,
        buffer_size=50000,
        max_episodes=num_episodes,
        epsilon_min=epsilon_min,
        epsilon_decay=epsilon_decay,
        epsilon_policy=ep_policy,
        reward_policy=RewardPolicyType.NONE,
        fc1_units=128,
        fc2_units=128,
        update_factor=0.005,
        update_target_network_method=UpdateTargetNetworkType.SOFT,
        target_update_frequency=50,
    )

    rewards = []
    for episode in range(num_episodes):
        obs = env.reset()
        total_reward = 0
        step = 0
        done = False

        while not done and step < max_steps_per_episode:
            action = agent.select_action(obs.reshape(1, -1))
            next_obs, reward, done, info = env.step(action)
            agent.store_experience(obs, next_obs, reward, action, done, huristic=None)
            loss = agent.train(episode)
            obs = next_obs
            total_reward += reward
            step += 1

        rewards.append(total_reward)
        print(
            f"Episode {episode + 1}/{num_episodes}, Total Reward: {total_reward}, "
            f"Epsilon: {agent.get_epsilon():.3f}, Loss: {loss}, Vehicles: {info.get('active_vehicles', 0)}"
        )

    env.close()

    import matplotlib.pyplot as plt

    plt.plot(rewards)
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.title("DQN Training on 4-Way SUMO Intersection")
    plt.show()
    agent.save("checkpoints/dqn_final")
    return rewards


if __name__ == "__main__":
    if not os.path.exists(Config.NET_FILE):
        print(f"[ERROR] Net file not found: {Config.NET_FILE}")
        sys.exit(1)
    env = SumoEnv(Config)
    rewards = train_dqn(env, num_episodes=EPOCHES)
    print("[INFO] Training complete.")


---

### Running `run.sh`

The `run.sh` script will automatically install all required dependencies, including SUMO .  
It is designed with the assumption that the target machine is running Ubuntu, so no additional setup should be necessary.


In [ ]:
!./run.sh